## Construction du dataset au niveau entité (df_entity) – Avril 2024

Objectif

L’objectif de cette étape est de construire un jeu de données au niveau entité (1 ligne = 1 entité WalletExplorer) à partir des transactions Bitcoin et des labels WalletExplorer. Ce dataset correspond au niveau “entité” utilisé comme baseline dans notre pipeline, avant de générer les datasets “adresse” et “motifs” nécessaires à l’approche en cascade.


Données utilisées
Nous utilisons deux types de fichiers :
Mapping WalletExplorer des adresses actives 2024 (active_2024_addresses.csv)
Ce fichier contient les colonnes :
category : catégorie de l’entité (exchanges, gambling, services-others, etc.)
wallet : nom/identifiant de l’entité (cluster WalletExplorer)
address : adresse Bitcoin appartenant à cette entité
last_block : dernier bloc où l’adresse est observée (information utile pour des filtrages ultérieurs)
Transactions Bitcoin journalières (blockchair_bitcoin_YYYYMMDD.csv)
Chaque fichier contient :
transaction_hash, time, block_id
inputs, outputs : chaînes de caractères représentant des listes de paires (adresse, montant)
les paires sont séparées par ;
l’adresse et le montant sont séparés par ,
Certaines pseudo-adresses du type d-... correspondent à des sorties scripts (ex. OP_RETURN) et sont ignorées car elles ne représentent pas de vraies adresses Bitcoin.
Méthode
1) Construction des mappings de labels
À partir de active_2024_addresses.csv, nous construisons :
un dictionnaire address_to_wallet qui associe chaque address à son wallet,
un dictionnaire wallet_to_label qui associe chaque wallet à sa category.
Ces mappings permettent d’identifier rapidement si une adresse observée dans une transaction appartient à une entité labellisée.
2) Parsing des inputs/outputs
Pour chaque transaction, les champs inputs et outputs sont convertis en listes de tuples :
[(addr_1, amount_1), (addr_2, amount_2), ...]
Les pseudo-adresses commençant par d- sont filtrées.
3) Agrégation des features au niveau entité
En parcourant toutes les transactions sur la fenêtre considérée, nous maintenons des accumulateurs par wallet afin de calculer :
amount_received : somme des montants reçus par les adresses du wallet (adresse du wallet en output)
amount_sent : somme des montants envoyés par les adresses du wallet (adresse du wallet en input)
balance = amount_received - amount_sent
n_tx_received : nombre de transactions où le wallet apparaît au moins une fois en output
n_tx_sent : nombre de transactions où le wallet apparaît au moins une fois en input
(les transactions sont comptées sans doublons grâce à un set de tx_hash par wallet)
Nous calculons également deux mesures de “degré” d’interaction externes :
n_addr_received_from : nombre d’adresses externes distinctes observées dans les inputs des transactions où le wallet reçoit
n_addr_sent_to : nombre d’adresses externes distinctes observées dans les outputs des transactions où le wallet envoie
Une adresse est considérée “externe” si elle n’appartient pas au même wallet selon address_to_wallet.
4) Création du dataframe final
Le dataframe df_entity contient une ligne par wallet avec les colonnes :
entity_id (= wallet), label (= category),
amount_received, amount_sent, balance,
n_tx_received, n_tx_sent,
n_addr_received_from, n_addr_sent_to.

In [5]:
import pandas as pd
import glob
from collections import defaultdict

# ===============================
# 1. Charger mapping entités
# ===============================

entities_path = "data_raw/entities/active_2024_addresses.csv"
entities_df = pd.read_csv(entities_path)

entities_df["address"] = entities_df["address"].astype(str).str.strip()
entities_df["wallet"] = entities_df["wallet"].astype(str).str.strip()
entities_df["category"] = entities_df["category"].astype(str).str.strip()

address_to_wallet = dict(zip(entities_df["address"], entities_df["wallet"]))
wallet_to_label = (
    entities_df.drop_duplicates("wallet")
    .set_index("wallet")["category"]
    .to_dict()
)

print("Nombre total d’adresses labellisées :", len(address_to_wallet))
print("Nombre total d’entités :", len(wallet_to_label))


# ===============================
# 2. Fonction parsing inputs/outputs
# ===============================

def parse_io_field(s):
    if pd.isna(s) or not str(s).strip():
        return []

    result = []
    for part in str(s).split(";"):
        part = part.strip()
        if "," not in part:
            continue

        addr, amt = part.split(",", 1)
        addr = addr.strip()

        if addr.startswith("d-"):  # ignorer pseudo-adresses
            continue

        try:
            amount = int(amt.strip())
        except:
            continue

        result.append((addr, amount))

    return result


# ===============================
# 3. Initialiser accumulateurs
# ===============================

received = defaultdict(int)
sent = defaultdict(int)

tx_recv = defaultdict(set)
tx_sent = defaultdict(set)

recv_from = defaultdict(set)
sent_to = defaultdict(set)


# ===============================
# 4. Charger toutes les transactions d’avril
# ===============================

tx_files = sorted(
    glob.glob("data_raw/transactions/2024/04/blockchair_bitcoin_202404*.csv")
)

print("Nombre de fichiers avril :", len(tx_files))

for path in tx_files:
    print("Traitement :", path)

    for chunk in pd.read_csv(path, chunksize=200000):

        for txh, ins, outs in zip(
            chunk["transaction_hash"],
            chunk["inputs"],
            chunk["outputs"]
        ):

            in_list = parse_io_field(ins)
            out_list = parse_io_field(outs)

            in_wallets = set()
            out_wallets = set()

            # INPUTS (sent)
            for addr, amt in in_list:
                wallet = address_to_wallet.get(addr)
                if wallet:
                    sent[wallet] += amt
                    in_wallets.add(wallet)

            # OUTPUTS (received)
            for addr, amt in out_list:
                wallet = address_to_wallet.get(addr)
                if wallet:
                    received[wallet] += amt
                    out_wallets.add(wallet)

            for w in in_wallets:
                tx_sent[w].add(txh)

            for w in out_wallets:
                tx_recv[w].add(txh)

            input_addrs = {a for a, _ in in_list}
            output_addrs = {a for a, _ in out_list}

            for w in out_wallets:
                for a in input_addrs:
                    if address_to_wallet.get(a) != w:
                        recv_from[w].add(a)

            for w in in_wallets:
                for a in output_addrs:
                    if address_to_wallet.get(a) != w:
                        sent_to[w].add(a)


# ===============================
# 5. Construire df_entity
# ===============================

wallets = list(wallet_to_label.keys())

df_entity = pd.DataFrame({
    "entity_id": wallets,
    "label": [wallet_to_label[w] for w in wallets],
    "amount_received": [received[w] for w in wallets],
    "amount_sent": [sent[w] for w in wallets],
    "balance": [received[w] - sent[w] for w in wallets],
    "n_tx_received": [len(tx_recv[w]) for w in wallets],
    "n_tx_sent": [len(tx_sent[w]) for w in wallets],
    "n_addr_received_from": [len(recv_from[w]) for w in wallets],
    "n_addr_sent_to": [len(sent_to[w]) for w in wallets],
})

print("\nRésumé :")
print(df_entity.describe())

# ===============================
# 6. Sauvegarde
# ===============================

output_path = "data_processed/2024_04/df_entity_2024_04.csv"
df_entity.to_csv(output_path, index=False)

print("\nFichier sauvegardé :", output_path)


Nombre total d’adresses labellisées : 408601
Nombre total d’entités : 73
Nombre de fichiers avril : 3
Traitement : data_raw/transactions/2024/04/blockchair_bitcoin_20240401.csv
Traitement : data_raw/transactions/2024/04/blockchair_bitcoin_20240402.csv
Traitement : data_raw/transactions/2024/04/blockchair_bitcoin_20240403.csv

Résumé :
       amount_received   amount_sent       balance  n_tx_received   n_tx_sent  \
count     7.300000e+01  7.300000e+01  7.300000e+01      73.000000   73.000000   
mean      3.436207e+10  3.443774e+10 -7.566850e+07     108.917808   10.013699   
std       2.858136e+11  2.857378e+11  1.396921e+09     497.668055   39.021181   
min       0.000000e+00  0.000000e+00 -1.113943e+10       0.000000    0.000000   
25%       0.000000e+00  0.000000e+00  0.000000e+00       0.000000    0.000000   
50%       0.000000e+00  0.000000e+00  0.000000e+00       0.000000    0.000000   
75%       5.478180e+05  0.000000e+00  0.000000e+00       1.000000    0.000000   
max       2.442

Construction du dataset au niveau entité (df_entity) – Avril 2024

In [6]:
import os

out_dir = "data_processed/2024_04"
os.makedirs(out_dir, exist_ok=True)

out_path = f"{out_dir}/df_entity_2024_04.csv"
df_entity.to_csv(out_path, index=False)

print("✅ Sauvé :", out_path)


✅ Sauvé : data_processed/2024_04/df_entity_2024_04.csv


In [7]:
(df_entity["n_tx_received"] > 0).sum(), (df_entity["n_tx_sent"] > 0).sum()


(np.int64(25), np.int64(15))

In [8]:
df_entity.sort_values("n_tx_received", ascending=False).head(10)[
    ["entity_id","label","n_tx_received","n_tx_sent","amount_received","amount_sent"]
]


,entity_id,label,n_tx_received,n_tx_sent,amount_received,amount_sent
26,Kraken.com,exchanges,3512,265,2442494278293,2441942268595
30,Luno.com,exchanges,2265,15,3117852245,14257284780
24,Huobi.com-2,exchanges,932,179,43571479982,41498562948
17,CoinSpot.com.au,exchanges,505,10,1956122696,2081990488
32,MercadoBitcoin.com.br,exchanges,221,83,890738265,844685701
35,Poloniex.com,exchanges,199,60,4420465552,4971084623
8,Bitcoin.de,exchanges,108,40,10956152664,7544359895
16,CoinMotion.com,exchanges,87,45,356144084,544850061
58,HolyTransaction.com,services-others,40,8,74111049,110600824
15,CoinHako.com,exchanges,27,0,471970600,0


In [9]:
active_recv = (df_entity["n_tx_received"] > 0).sum()
active_sent = (df_entity["n_tx_sent"] > 0).sum()
active_any = ((df_entity["n_tx_received"] + df_entity["n_tx_sent"]) > 0).sum()

print("Entités actives (reçoivent):", active_recv)
print("Entités actives (envoient):", active_sent)
print("Entités actives (au moins une tx):", active_any)


Entités actives (reçoivent): 25
Entités actives (envoient): 15
Entités actives (au moins une tx): 25


In [10]:
df_entity.assign(active=((df_entity["n_tx_received"] + df_entity["n_tx_sent"]) > 0)) \
         .groupby(["label","active"])["entity_id"].count() \
         .unstack(fill_value=0) \
         .sort_values(True, ascending=False)


active,False,True
label,,
exchanges,26,14
services-others,11,8
gambling,2,3
old-historic,5,0
pools,4,0


In [11]:
import pandas as pd
import glob

tx_files = sorted(glob.glob("../data_raw/transactions/2024/04/blockchair_bitcoin_202404*.csv"))

def parse_addrs_only(s):
    if pd.isna(s) or not str(s).strip():
        return []
    addrs = []
    for part in str(s).split(";"):
        part = part.strip()
        if "," not in part:
            continue
        addr = part.split(",", 1)[0].strip()
        if addr.startswith("d-"):
            continue
        addrs.append(addr)
    return addrs

total_txs = 0
txs_with_labeled = 0

for path in tx_files:
    for chunk in pd.read_csv(path, usecols=["transaction_hash","inputs","outputs"], chunksize=200000):
        for ins, outs in zip(chunk["inputs"], chunk["outputs"]):
            total_txs += 1
            in_addrs = parse_addrs_only(ins)
            out_addrs = parse_addrs_only(outs)
            if any(a in address_to_wallet for a in in_addrs) or any(a in address_to_wallet for a in out_addrs):
                txs_with_labeled += 1

print("Total txs :", total_txs)
print("Txs touchant au moins une adresse labellisée :", txs_with_labeled)
print("Couverture (%) :", 100.0 * txs_with_labeled / max(total_txs, 1))


Total txs : 0
Txs touchant au moins une adresse labellisée : 0
Couverture (%) : 0.0


Contrôle qualité
Plusieurs sanity checks sont effectués :
nombre d’entités actives sur la fenêtre (au moins une transaction),
répartition actives/inactives par catégorie,
couverture : proportion de transactions touchant au moins une adresse labellisée (attendue faible sur Bitcoin).
Sauvegarde
Le dataframe est exporté pour réutilisation dans les étapes suivantes (adresse, motifs, cascade) :
data_processed/2024_04/df_entity_2024_04.csv

In [13]:
import pandas as pd
import glob

tx_files = [
    "data_raw/transactions/2024/04/blockchair_bitcoin_20240401.csv",
    "data_raw/transactions/2024/04/blockchair_bitcoin_20240402.csv",
    "data_raw/transactions/2024/04/blockchair_bitcoin_20240403.csv",
]

def parse_addrs_only(s):
    if pd.isna(s) or not str(s).strip():
        return []
    addrs = []
    for part in str(s).split(";"):
        part = part.strip()
        if "," not in part:
            continue
        addr = part.split(",", 1)[0].strip()
        if addr.startswith("d-"):
            continue
        addrs.append(addr)
    return addrs

all_addresses = set()
labeled_addresses_seen = set()

for path in tx_files:
    for chunk in pd.read_csv(path, usecols=["inputs","outputs"], chunksize=200000):
        for ins, outs in zip(chunk["inputs"], chunk["outputs"]):
            in_addrs = parse_addrs_only(ins)
            out_addrs = parse_addrs_only(outs)

            for a in in_addrs + out_addrs:
                all_addresses.add(a)
                if a in address_to_wallet:
                    labeled_addresses_seen.add(a)

print("Total adresses uniques observées :", len(all_addresses))
print("Adresses labellisées observées :", len(labeled_addresses_seen))
print("Proportion (%) :", 100 * len(labeled_addresses_seen) / len(all_addresses))


Total adresses uniques observées : 2450370
Adresses labellisées observées : 8853
Proportion (%) : 0.3612923762533821


Sur la fenêtre du 1er au 3 avril 2024, nous observons X adresses uniques dans les transactions.
Parmi celles-ci, Y adresses appartiennent à des entités labellisées WalletExplorer, soit 0.36% du réseau observé.
Cela montre que notre dataset couvre une fraction limitée mais ciblée de l’activité Bitcoin, principalement centrée sur des entités identifiées (exchanges, services, gambling, etc.).